## 7.1 Simple RNN实现 - 模型搭建

#### 1. 为什么我们现在要学习 PyTorch 中的 RNN 实现？

##### 1.1 前面我们已经学了“原理层”的内容

到目前为止，我们已经理解了：

* RNN 的前向传播
* RNN 的反向传播 BPTT
* RNN 的参数更新过程

这些内容帮助我们从数学和计算图角度理解了 RNN 到底是怎么工作的。🧠

但是在实际开发中，我们一般不会手写：

* $W_{xh}$
* $W_{hh}$
* 每一个时间步的循环计算
* BPTT 的反向传播细节

因为 PyTorch 已经帮我们封装好了。

##### 1.2 所以现在要进入“工程实现层”

也就是说，我们现在要解决的问题变成：

> 在 PyTorch 中，RNN 模型到底怎么搭？

这部分要重点搞清楚：

* `nn.RNN` 是什么
* 它的输入张量长什么样
* 它的输出张量长什么样
* 如何把 RNN 和全连接层拼起来，变成一个完整模型

##### 1.3 这一节的目标

这一节我们先不急着上训练代码，也不急着做复杂案例。

我们只专注一件事：

> 学会用 PyTorch 正确搭建一个最基础的 Simple RNN 模型。✅

#### 2. PyTorch 中 RNN 的核心模块是什么？

##### 2.1 最基础的模块：`nn.RNN`

在 PyTorch 中，最基础的循环神经网络模块是：

`nn.RNN(...)`

它对应的就是我们前面学习的普通 RNN / Simple RNN。

也就是说，它内部做的事情，本质上就是在每个时间步反复执行：

$a_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h$

$h_t = \phi(a_t)$

其中激活函数默认是：

* `tanh`

也可以改成：

* `relu`

##### 2.2 它和我们前面学的公式是对应的

我们前面手动写的 RNN，核心是：

$a_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h$

$h_t = \phi(a_t)$

而 `nn.RNN` 就是把这整套过程帮我们封装好了。

所以你可以把它理解成：

> 一个可以自动按时间步处理序列数据的“循环层”。

#### 3. 先理解 RNN 输入数据的形状

##### 3.1 RNN 处理的不是单个样本，而是“序列”

普通全连接层常见输入是：

`[batch_size, input_features]`

而 RNN 不一样，因为它处理的是一段序列，所以输入中必须体现：

* batch 维度
* 时间步维度
* 每个时间步的特征维度

所以 RNN 输入本质上是一个三维张量。📦

##### 3.2 RNN 输入的三个维度

如果使用 `batch_first=True`，那么输入形状是：

`[batch_size, seq_len, input_size]`

分别表示：

* `batch_size`：一批有多少个样本
* `seq_len`：每个样本有多少个时间步
* `input_size`：每个时间步输入特征数

##### 3.3 举一个最简单的例子

假设我们现在有：

* 一批数据中有 $4$ 个样本
* 每个样本是长度为 $5$ 的序列
* 每个时间步有 $3$ 个特征

那么输入张量形状就是：

`[4, 5, 3]`

你可以把它理解为：

* $4$ 条序列
* 每条序列有 $5$ 个时间步
* 每个时间步输入一个 $3$ 维向量

##### 3.4 为什么这个形状很重要？

因为后面你在写代码时，最容易出错的地方就是：

> 输入维度搞错。

很多初学者会分不清：

* 哪一维是 batch
* 哪一维是时间步
* 哪一维是特征

#### 4. `nn.RNN` 的几个核心参数

##### 4.1 基本写法

最基础的写法如下：

```python
nn.RNN(
    input_size,
    hidden_size,
    num_layers=1,
    batch_first=True
)
```

##### 4.2 `input_size`

它表示：

> 每个时间步输入向量的特征数。

比如：

* 一个单词经过 embedding 后是 $100$ 维
* 一个时间步输入是 $8$ 个传感器数值
* 一天的天气记录有 $6$ 个指标

那这个数字就是 `input_size`。

例如：

`input_size = 3`

表示每个时间步输入 $3$ 个特征。

##### 4.3 `hidden_size`

它表示：

> 隐藏状态 $h_t$ 的维度大小。

也就是每个时间步输出的隐藏向量长度。

例如：

`hidden_size = 16`

表示每个时间步内部的隐藏状态是一个 $16$ 维向量。

这个参数非常重要，因为它决定了模型的“表示能力”。

你可以简单理解为：

* 越大，模型能表达的信息越多
* 但参数也会更多，训练成本也会更高

##### 4.4 `num_layers`

它表示：

> RNN 层堆叠的层数。

例如：

* `num_layers = 1` 表示单层 RNN
* `num_layers = 2` 表示两层 RNN 叠加

当前我们先学最基础版本，所以先使用：

`num_layers = 1`

##### 4.5 `batch_first=True`

这个参数非常常用。

如果写成：

`batch_first=True`

那么输入输出张量都会采用：

`[batch_size, seq_len, feature]`

这和我们平时处理 batch 的直觉更一致，所以在学习阶段很推荐打开。✅

如果不写这个参数，默认格式会变成：

`[seq_len, batch_size, feature]`

##### 4.6 `nonlinearity`

普通 RNN 的激活函数可以选：

* `"tanh"`（默认）
* `"relu"`

例如：

```python
nn.RNN(..., nonlinearity="tanh")
```

#### 5. `nn.RNN` 的输出到底是什么？

##### 5.1 调用 RNN 层时会返回两个结果

假设我们这样写：

```python
output, hn = self.rnn(x)
```

##### 5.2 `output` 是什么？

`output` 表示：

> 每一个时间步的隐藏状态汇总，注意并不是最终输出，而是隐藏状态。

如果：

* `batch_first=True`
* 单向 RNN
* 单层 RNN

那么它的形状是：

`[batch_size, seq_len, hidden_size]`

也就是说：

> 对于序列中的每个时间步，RNN 都会输出一个隐藏向量。

##### 5.3 `hn` 是什么？

`hn` 表示：

> 每一层的最后一个时间步的隐藏状态，注意这并不是最终输出，而是隐藏状态。

如果是单层单向 RNN，那么它的形状是：

`[num_layers, batch_size, hidden_size]`

如果只有 $1$ 层，那么其实就是：

`[1, batch_size, hidden_size]`

所以 `hn` 本质上就是：

> 最终时刻的隐藏状态汇总结果。

##### 5.4 `output` 和 `hn` 的关系

对于单层单向 RNN 来说：

* `output[:, -1, :]`：表示最后一个时间步的输出
* `hn[-1]`：表示最后一层的最终隐藏状态

在这种最简单情况下，它们本质上对应的是同一份最后时刻信息。

所以很多时候都可以用来做分类任务的最终特征。

#### 6. 什么时候用 `output`，什么时候用 `hn`？

##### 6.1 如果是 many-to-many 任务

比如序列标注任务中，每个时间步都要输出一个结果：

$x_1, x_2, x_3 \rightarrow \hat{y}_1, \hat{y}_2, \hat{y}_3$

这时通常需要使用：

* `output`

因为你要保留所有时间步的信息。

##### 6.2 如果是 many-to-one 任务

比如文本分类、情感分类：

$x_1, x_2, x_3, \dots, x_T \rightarrow \hat{y}$

这时通常只需要整个序列最后汇总出来的表示，所以常用：

* `output[:, -1, :]`
* 或 `hn[-1]`

我们后面的简单分类示例，就会采用这种方式。

#### 7. 如何把 RNN 搭成一个完整模型？

##### 7.1 只有 RNN 层还不够

`nn.RNN` 只负责做一件事：

> 从输入序列中提取时序特征。

但如果我们最终要做分类，比如二分类或多分类，还需要再接一个输出层。

所以一个最简单的 RNN 分类模型，一般结构是：

输入序列 $\rightarrow$ RNN 层 $\rightarrow$ 取最后时间步特征 $\rightarrow$ 全连接层 $\rightarrow$ 输出结果

##### 7.2 最简单结构示意

例如：

$x \rightarrow RNN \rightarrow h_{last} \rightarrow Linear \rightarrow \hat{y}$

这里：

* `RNN` 负责提取序列特征
* `Linear` 负责把隐藏特征映射

#### 8. 最基础的 PyTorch RNN 模型类

##### 8.1 代码结构

下面是一个最基础的 Simple RNN 分类模型：

In [1]:
import torch 
import torch.nn as nn

class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        # Define the RNN layer
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        # Define the output layer
        self.fc = nn.Linear(
            in_features=hidden_size,
            out_features=output_size
        )

    def forward(self, x):
        # Pass the input through the RNN layer
        rnn_out, _ = self.rnn(x)
        # Take the output from the last time step
        last_time_step = rnn_out[:, -1, :]
        # Pass the last time step output through the fully connected layer
        output = self.fc(last_time_step)
        return output

##### 8.2 init 中做了什么？

在初始化方法中，我们定义了两层：

第一层：
``` python
self.rnn = nn.RNN(…)
```
表示 RNN 层，用来处理序列。

第二层：
``` python
self.fc = nn.Linear(hidden_size, num_classes)
```
表示全连接输出层。

这里的意思是：
* 输入是最后时间步的隐藏状态，维度是 hidden_size
* 输出是类别数，维度是 num_classes

##### 8.3 forward() 中做了什么？
在前向传播中：
``` python
output, hn = self.rnn(x)
```
先把输入序列送入 RNN。

此时：
* output 是所有时间步输出
* hn 是最后隐藏状态

然后：
``` python
last_output = output[:, -1, :]
```
取最后一个时间步的输出。

最后：
``` python
out = self.fc(last_output)
```
送入全连接层得到最终结果。

#### 9. 为什么这里取最后一个时间步？

##### 9.1 因为这是最经典的 many-to-one 结构

如果一个序列最终只输出一个结果，例如：

* 句子情感分类
* 股票趋势分类
* 序列整体类别判断

那么通常会认为：

> 最后一个时间步的隐藏状态，已经汇总了前面整个序列的信息。

所以我们取：

`output[:, -1, :]`

作为整个序列的表示。

##### 9.2 这和前面学的理论是对应的

前面我们学过：

* $h_t$ 会不断接收：
  * 当前输入 $x_t$
  * 前一个隐藏状态 $h_{t-1}$

所以到了最后一个时间步 $h_T$ 时，它理论上已经整合了整条序列的信息。

因此用它来做最终分类，是很自然的设计。


#### 10. 输入输出维度示例

##### 10.1 假设模型参数如下

```python
input_size = 3
hidden_size = 8
num_layers = 1
num_classes = 2
```
并且输入数据（batch_first=True）：

`x.shape = [4, 5, 3]`

表示：
* batch size = 4
* sequence length = 5
* each time step has 3 features

##### 10.2 经过 RNN 后
``` python
output, hn = self.rnn(x)
```
输出形状为：

`output.shape = [4, 5, 8]`

因为：
* 4 个样本
* 每个样本 5 个时间步
* 每个时间步输出 8 维隐藏状态

每一层RNN最后一个隐藏状态的形状为：

`hn.shape = [number_layer, 4, 8]`

因为：
* number_layer 层 RNN
* 4 个样本
* 最终隐藏状态维度 8

##### 10.3 取最后一个时间步后

`last_output = output[:, -1, :]`

形状变成：

`[4, 8]`

表示：
* 4 个样本
* 每个样本一个 8 维最终特征向量

##### 10.4 再经过全连接层
``` python
out = self.fc(last_output)
```
如果 `num_classes = 2`，则输出形状为：

`[4, 2]`

表示：
* 4 个样本
* 每个样本输出 2 个类别分数